# First-Diff 位置分布分析

分析 TIGER beam search 预测与 target SID 的第一次分叉发生在哪个 token 位置。

In [1]:
import sys
sys.path.insert(0, '../model')

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from dataset import GenRecDataset
from dataloader import GenRecDataLoader
from main import TIGER

# ── Config ──
CKPT_PATH = '../model/ckpt/Beauty_baseline_ce/Jul-09-2026_14-20-11/best_model.pth'
# CKPT_PATH = '../model/ckpt/Beauty_baseline_ce_rqkmeans/Jul-09-2026_21-23-03/best_model.pth'
CODE_PATH = '../data/Beauty/Beauty_t5_rqvae_260709-sk[0-0-0.003].npy'
# CODE_PATH = '../data/Beauty/Beauty_kmeans_code.npy'
DATASET_PATH = '../data/Beauty/test.parquet'
BEAM_SIZE = 20
INFER_SIZE = 96
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")
print(f"CKPT: {CKPT_PATH}")
print(f"CODE: {CODE_PATH}")

Device: cuda
CKPT: ../model/ckpt/Beauty_baseline_ce/Jul-09-2026_14-20-11/best_model.pth
CODE: ../data/Beauty/Beauty_t5_rqvae_260709-sk[0-0-0.003].npy


In [2]:
# ── Load model ──
config = {
    'num_layers': 4, 'num_decoder_layers': 4,
    'd_model': 128, 'd_ff': 1024, 'num_heads': 6, 'd_kv': 64,
    'dropout_rate': 0.1, 'vocab_size': 1025,
    'pad_token_id': 0, 'eos_token_id': 0,
    'feed_forward_proj': 'relu',
}

model = TIGER(config)
state_dict = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict, strict=True)
model.to(DEVICE)
model.eval()

# ── Data ──
test_ds = GenRecDataset(DATASET_PATH, CODE_PATH, mode='evaluation', max_len=20)
test_loader = GenRecDataLoader(test_ds, batch_size=INFER_SIZE, shuffle=False)
print(f"Test samples: {len(test_ds)}")

# SID → item mapping (for hit checking)
def offset_to_raw_sid(offset_tokens, codebook_size=256):
    return tuple(int(t) - 1 - layer * codebook_size
                 for layer, t in enumerate(offset_tokens))

raw_codes = np.load(CODE_PATH)
sid_to_item = {}
for item_id, sid in enumerate(raw_codes):
    sid_to_item[tuple(int(s) for s in sid)] = item_id + 1

Test samples: 22363


In [3]:
# ── Run inference ──
top1_first_diff = []     # int: 0=L1, 1=L2, 2=L3, 3=L4, -1=exact
all_beam_diff = []       # flattened for all beams (path-level, kept for distribution)
top1_hit = []            # bool: top-1 SID exactly matches target
any_beam_exact = []      # bool: ANY beam has exact SID match (sample-level)
any_hit_20 = []          # bool: target item in top-20 unique items

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Analysing", ncols=90):
        input_ids = batch['history'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['target'].cpu().numpy()  # (B, 4) offset tokens

        result = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=BEAM_SIZE,
            output_scores=True,
            return_dict_in_generate=True,
        )
        sequences = result.sequences[:, 1:].cpu().numpy()  # (B*beam, 4)
        sequences = sequences.reshape(input_ids.shape[0], BEAM_SIZE, -1)

        for i in range(input_ids.shape[0]):
            target_sid = labels[i]
            target_raw = offset_to_raw_sid(target_sid)
            target_item = sid_to_item.get(target_raw, None)

            seen_items = set()
            sample_exact = False
            for j in range(BEAM_SIZE):
                pred_sid = sequences[i, j]
                pred_raw = offset_to_raw_sid(pred_sid)
                pred_item = sid_to_item.get(pred_raw, None)
                if pred_item is not None and pred_item not in seen_items:
                    seen_items.add(pred_item)

                # First-diff position
                diff_pos = -1
                for pos in range(4):
                    if pred_sid[pos] != target_sid[pos]:
                        diff_pos = pos
                        break
                all_beam_diff.append(diff_pos)
                if diff_pos == -1:
                    sample_exact = True

                if j == 0:
                    top1_first_diff.append(diff_pos)
                    top1_hit.append(diff_pos == -1)

            any_beam_exact.append(sample_exact)
            any_hit_20.append(
                target_item is not None and target_item in seen_items
            )

top1_first_diff = np.array(top1_first_diff, dtype=np.int8)
all_beam_diff = np.array(all_beam_diff, dtype=np.int8)
top1_hit = np.array(top1_hit, dtype=bool)
any_beam_exact = np.array(any_beam_exact, dtype=bool)
any_hit_20 = np.array(any_hit_20, dtype=bool)
N = len(top1_first_diff)
print(f"Done. {N} samples analysed.")

Analysing: 100%|████████████████████████████████████████| 233/233 [00:23<00:00,  9.94it/s]

Done. 22363 samples analysed.


## 1. Top-1 首次分叉位置分布

最强预测（beam rank 1）在哪一层开始偏离 target？

In [4]:
labels_map = {0: 'L1', 1: 'L2', 2: 'L3', 3: 'L4/pad', -1: 'EXACT MATCH'}
print(f"{'Position':>12s}  {'Count':>8s}  {'Pct':>7s}  Bar")
print("-" * 50)
for pos in [-1, 0, 1, 2, 3]:
    count = (top1_first_diff == pos).sum()
    pct = count / N * 100
    bar = '█' * int(pct * 2)
    print(f"{labels_map[pos]:>12s}: {count:8d} ({pct:5.1f}%) {bar}")

print(f"\nTop-1 exact match rate: {top1_hit.mean()*100:.2f}%")
print(f"Any hit in top-20:     {any_hit_20.mean()*100:.2f}%")

    Position     Count      Pct  Bar
--------------------------------------------------
 EXACT MATCH:      236 (  1.1%) ██
          L1:    19046 ( 85.2%) ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
          L2:     2835 ( 12.7%) █████████████████████████
          L3:      246 (  1.1%) ██
      L4/pad:        0 (  0.0%) 

Top-1 exact match rate: 1.06%
Any hit in top-20:     8.14%


## 2. Miss 样本的首次分叉位置

top-20 都没命中 target 的样本，top-1 错在哪？

In [5]:
miss_mask = ~any_hit_20
n_miss = miss_mask.sum()
if n_miss > 0:
    print(f"Top-20 miss: {n_miss} samples ({n_miss/N*100:.1f}%)")
    print(f"\n{'Position':>12s}  {'Count':>8s}  {'Pct':>7s}")
    print("-" * 35)
    for pos in [-1, 0, 1, 2, 3]:
        count = (top1_first_diff[miss_mask] == pos).sum()
        pct = count / n_miss * 100
        print(f"{labels_map[pos]:>12s}: {count:8d} ({pct:5.1f}%)")
else:
    print("All samples hit in top-20.")

Top-20 miss: 20543 samples (91.9%)

    Position     Count      Pct
-----------------------------------
 EXACT MATCH:        0 (  0.0%)
          L1:    18139 ( 88.3%)
          L2:     2280 ( 11.1%)
          L3:      124 (  0.6%)
      L4/pad:        0 (  0.0%)


## 3. 条件正确率（Error Propagation）

L1 正确时，L2 还能正确吗？L1+L2 都对时，L3 还对吗？

In [6]:
# top-1 beam
l1_ok = top1_first_diff != 0                             # L1 correct (error not at L1)
l2_ok = (top1_first_diff != 1) & l1_ok                  # L2 correct (L1 ok AND error not at L2)
l3_ok = (top1_first_diff != 2) & l2_ok                  # L3 correct (L1+L2 ok AND error not at L3)
l4_ok = top1_first_diff == -1                            # L4 correct = exact match

print("Top-1 beam conditional accuracy:")
print(f"  P(L1 correct):                    {l1_ok.mean()*100:.2f}%")
print(f"  P(L2 correct | L1 correct):       {(l2_ok.sum()/l1_ok.sum()*100) if l1_ok.sum()>0 else 0:.2f}%")
print(f"  P(L3 correct | L1+L2 correct):    {(l3_ok.sum()/l2_ok.sum()*100) if l2_ok.sum()>0 else 0:.2f}%")
print(f"  P(L4 correct | L1+L2+L3 correct): {(l4_ok.sum()/l3_ok.sum()*100) if l3_ok.sum()>0 else 0:.2f}%")

# sanity check
assert (l1_ok.sum() == (top1_first_diff != 0).sum())
assert (l2_ok.sum() == ((top1_first_diff == -1) | (top1_first_diff == 2) | (top1_first_diff == 3)).sum())
assert (l3_ok.sum() == ((top1_first_diff == -1) | (top1_first_diff == 3)).sum())
assert (l4_ok.sum() == (top1_first_diff == -1).sum())
print("  ✓ sanity checks passed")

Top-1 beam conditional accuracy:
  P(L1 correct):                    14.83%
  P(L2 correct | L1 correct):       14.53%
  P(L3 correct | L1+L2 correct):    48.96%
  P(L4 correct | L1+L2+L3 correct): 100.00%
  ✓ sanity checks passed


## 4. Beam Search 的样本级增益

对比 top-1 vs 全部 20 beam（样本级）：beam search 能否在非 top-1 的 beam 中找到 exact match？

In [7]:
print(f"{'Metric':>35s}  {'Rate':>8s}")
print("-" * 46)
print(f"{'Top-1 exact (SID-level)':>35s}: {top1_hit.mean()*100:7.2f}%")
print(f"{'Any-beam exact (SID-level)':>35s}: {any_beam_exact.mean()*100:7.2f}%")
print(f"{'Any-beam hit (item-level, dedup)':>35s}: {any_hit_20.mean()*100:7.2f}%")

# Show exact matches found only in beams 2-20 (missed by top-1)
only_in_beam = any_beam_exact & ~top1_hit
print(f"\nSamples with exact match ONLY in beams 2-20: {only_in_beam.sum()} ({only_in_beam.mean()*100:.2f}%)")

# ── First-diff position distribution: Top-1 vs path-level vs sample-level ──
all_beam_2d = all_beam_diff.reshape(N, BEAM_SIZE)  # (N, BEAM_SIZE) for sample-level aggregation

print(f"\n{'Position':>12s}  {'Top-1':>8s}  {'Path-level':>11s}  {'Sample-level':>13s}")
print("-" * 50)
for pos in [-1, 0, 1, 2, 3]:
    top1_pct = (top1_first_diff == pos).sum() / N * 100
    path_pct = (all_beam_diff == pos).sum() / len(all_beam_diff) * 100
    sample_pct = (all_beam_2d == pos).any(axis=1).mean() * 100
    print(f"{labels_map[pos]:>12s}: {top1_pct:7.2f}%  {path_pct:10.2f}%  {sample_pct:12.2f}%")

print(f"\n  Top-1:        top-1 beam only, denominator = N")
print(f"  Path-level:   all N×{BEAM_SIZE} beam-paths, denominator = N×{BEAM_SIZE}")
print(f"  Sample-level:  any of the {BEAM_SIZE} beams, denominator = N")

                             Metric      Rate
----------------------------------------------
            Top-1 exact (SID-level):    1.06%
         Any-beam exact (SID-level):    8.14%
   Any-beam hit (item-level, dedup):    8.14%

Samples with exact match ONLY in beams 2-20: 1584 (7.08%)

    Position     Top-1   Path-level   Sample-level
--------------------------------------------------
 EXACT MATCH:    1.06%        0.41%          8.14%
          L1:   85.17%       85.83%         99.33%
          L2:   12.68%       12.86%         43.11%
          L3:    1.10%        0.91%          7.48%
      L4/pad:    0.00%        0.00%          0.07%

  Top-1:        top-1 beam only, denominator = N
  Path-level:   all N×20 beam-paths, denominator = N×20
  Sample-level:  any of the 20 beams, denominator = N


## 5. Beam Survival：逐层 Recall@K

对每个 K（取前 K 条 beam），检查任意一条 beam 是否在 L1/L1+L2/L1+L2+L3/Full 层级正确。

这直接决定了「L1 是不是限制 Recall@20 的真正瓶颈」。

In [8]:
# ── Beam Survival Table ──
# For each K, what fraction of samples have the correct prefix in ANY of the top-K beams?
all_beam_2d = all_beam_diff.reshape(N, BEAM_SIZE)  # (N, 20)

# Per-beam correctness at each depth
l1_beam   = (all_beam_2d != 0)                                  # L1 token correct
l12_beam  = (all_beam_2d == -1) | (all_beam_2d > 1)            # L1+L2 correct
l123_beam = (all_beam_2d == -1) | (all_beam_2d > 2)            # L1+L2+L3 correct
full_beam = (all_beam_2d == -1)                                  # all 4 correct

ks = [1, 5, 10, 20]
print("Beam Survival: R_depth @ K  (sample-level, any beam in top-K)")
print()
header = f"{'Level':>12s}  " + "  ".join([f"K={k:<2d}   " for k in ks])
print(header)
print("-" * len(header))
for name, mask in [("L1", l1_beam), ("L1+L2", l12_beam),
                    ("L1+L2+L3", l123_beam), ("Full (exact)", full_beam)]:
    vals = [mask[:, :k].any(axis=1).mean() * 100 for k in ks]
    print(f"{name:>12s}:  " + "  ".join([f"{v:5.2f}%" for v in vals]))

# ── Drop-off analysis ──
print(f"\nDrop-off (K=20 → K=1):")
for name, mask in [("L1", l1_beam), ("L1+L2", l12_beam),
                    ("L1+L2+L3", l123_beam), ("Full", full_beam)]:
    r1  = mask[:, :1].any(axis=1).mean() * 100
    r20 = mask[:, :20].any(axis=1).mean() * 100
    print(f"  {name:>12s}: {r20:5.2f}% → {r1:5.2f}%  (Δ = {r20 - r1:5.2f}pp)")

# ── Gap analysis: what limits Full@20? ──
r_l1_20    = l1_beam[:, :20].any(axis=1).mean()
r_l12_20   = l12_beam[:, :20].any(axis=1).mean()
r_l123_20  = l123_beam[:, :20].any(axis=1).mean()
r_full_20  = full_beam[:, :20].any(axis=1).mean()

print(f"\nSurvival chain @ K=20:")
print(f"  L1:       {r_l1_20*100:5.2f}%")
print(f"  L1+L2:    {r_l12_20*100:5.2f}%  (conditional on L1: {r_l12_20/r_l1_20*100:5.2f}%)")
print(f"  L1+L2+L3: {r_l123_20*100:5.2f}%  (conditional on L1+L2: {r_l123_20/r_l12_20*100:5.2f}%)")
print(f"  Full:     {r_full_20*100:5.2f}%  (conditional on L1+L2+L3: {r_full_20/r_l123_20*100:5.2f}%)")

Beam Survival: R_depth @ K  (sample-level, any beam in top-K)

       Level  K=1      K=5      K=10     K=20   
------------------------------------------------
          L1:  14.83%  29.04%  37.03%  45.25%
       L1+L2:   2.16%   6.15%   9.10%  13.02%
    L1+L2+L3:   1.06%   3.50%   5.60%   8.17%
Full (exact):   1.06%   3.48%   5.59%   8.14%

Drop-off (K=20 → K=1):
            L1: 45.25% → 14.83%  (Δ = 30.42pp)
         L1+L2: 13.02% →  2.16%  (Δ = 10.87pp)
      L1+L2+L3:  8.17% →  1.06%  (Δ =  7.12pp)
          Full:  8.14% →  1.06%  (Δ =  7.08pp)

Survival chain @ K=20:
  L1:       45.25%
  L1+L2:    13.02%  (conditional on L1: 28.77%)
  L1+L2+L3:  8.17%  (conditional on L1+L2: 62.77%)
  Full:      8.14%  (conditional on L1+L2+L3: 99.56%)
